# Golf Modeling - Data Exploration
Summary tables for each processed CSV: field definitions, data types, null rates, outliers, and quality issues.

In [13]:
import pandas as pd
import numpy as np
from IPython.display import display

DATA = "../data/processed/"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)


def outlier_count(series):
    """IQR x1.5 for numeric; rare-value count (<0.5% freq) for categorical."""
    s = series.dropna()
    if pd.api.types.is_numeric_dtype(s):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            return 0
        return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    else:
        freq = s.value_counts(normalize=True)
        rare = freq[freq < 0.005].index
        return int(s.isin(rare).sum())


def summary_table(df, meta):
    n = len(df)
    rows = []
    for col in df.columns:
        m = meta.get(col, {})
        null_n = int(df[col].isnull().sum())
        out_n  = outlier_count(df[col])
        dtype  = df[col].dtype
        if pd.api.types.is_float_dtype(dtype):
            t = "float"
        elif pd.api.types.is_integer_dtype(dtype):
            t = "integer"
        elif pd.api.types.is_bool_dtype(dtype):
            t = "boolean"
        else:
            t = "string"
        rows.append({
            "Field":               col,
            "Description":         m.get("description", ""),
            "Type":                t,
            "Blanks":              null_n,
            "% Blank":             round(null_n / n, 6),
            "Variable Category":   m.get("category", ""),
            "Outliers":            out_n,
            "% Outlier":           round(out_n / n, 6),
            "Data Quality Issues": m.get("quality_issues", ""),
        })
    out = pd.DataFrame(rows)
    apa_styles = [
        {"selector": "", "props": [("border-collapse", "collapse")]},
        {"selector": "thead tr th", "props": [
            ("border-top", "2px solid black"),
            ("border-bottom", "1px solid black"),
            ("border-left", "none"),
            ("border-right", "none"),
            ("background-color", "white"),
            ("color", "black"),
            ("text-align", "center"),
            ("padding", "4px 8px"),
        ]},
        {"selector": "tbody td", "props": [
            ("border", "none"),
            ("background-color", "white"),
            ("color", "black"),
            ("text-align", "center"),
            ("padding", "4px 8px"),
        ]},
        {"selector": "tbody tr:last-child td", "props": [
            ("border-bottom", "2px solid black"),
        ]},
    ]
    return out.style \
        .format({"% Blank": "{:.2%}", "% Outlier": "{:.2%}"}) \
        .set_table_styles(apa_styles) \
        .hide(axis="index")


def section(df, meta, title):
    print(f"\n{'='*80}\n  {title}  ({len(df.columns)} fields, {len(df):,} rows)\n{'='*80}")
    display(summary_table(df, meta))

## 1. players.csv

In [14]:
players = pd.read_csv(DATA + "players.csv")
players.head()

,dg_id,player_name,country,country_code,amateur
0,30,"Calcavecchia, Mark",United States,USA,0
1,44,"Couples, Fred",United States,USA,0
2,72,"Funk, Fred",United States,USA,0
3,87,"Haas, Jay",United States,USA,0
4,94,"Hart, Jeff",United States,USA,0


In [15]:
players_meta = {
    "dg_id": {
        "description": "DataGolf player ID (primary key)",
        "category": "identifier",
        "quality_issues": "",
    },
    "player_name": {
        "description": "Player name in 'Last, First' format",
        "category": "identifier",
        "quality_issues": "Name format does not match winner field in events.csv, which appends DG ID in parentheses.",
    },
    "country": {
        "description": "Home country (full name)",
        "category": "demographic",
        "quality_issues": "",
    },
    "country_code": {
        "description": "ISO 3-letter country code",
        "category": "demographic",
        "quality_issues": "",
    },
    "amateur": {
        "description": "Amateur status flag (1 = amateur, 0 = pro)",
        "category": "demographic",
        "quality_issues": "Static snapshot only - does not reflect historical status. A now-professional player will show 0 even for rounds played as an amateur.",
    },
}

section(players, players_meta, "players.csv")


  players.csv  (5 fields, 3,438 rows)


Field,Description,Type,Blanks,% Blank,Variable Category,Outliers,% Outlier,Data Quality Issues
dg_id,DataGolf player ID (primary key),integer,0,0.00%,identifier,13,0.38%,
player_name,"Player name in 'Last, First' format",string,0,0.00%,identifier,3438,100.00%,"Name format does not match winner field in events.csv, which appends DG ID in parentheses."
country,Home country (full name),string,0,0.00%,demographic,272,7.91%,
country_code,ISO 3-letter country code,string,0,0.00%,demographic,272,7.91%,
amateur,"Amateur status flag (1 = amateur, 0 = pro)",integer,0,0.00%,demographic,0,0.00%,Static snapshot only - does not reflect historical status. A now-professional player will show 0 even for rounds played as an amateur.


In [16]:
players.isna().sum()

dg_id           0
player_name     0
country         0
country_code    0
amateur         0
dtype: int64

## 2. courses.csv

In [17]:
courses = pd.read_csv(DATA + "courses.csv")
courses.head()

,course_key,course_name,location,country,latitude,longitude
0,2,Indian Wells CC,"Indian Wells, CA",USA,33.7175,-116.3411
1,4,Torrey Pines GC (South),"San Diego, CA",USA,32.9050,-117.2450
2,5,Pebble Beach GL,"Pebble Beach, CA",USA,36.5690,-121.9510
3,6,Waialae CC,"Honolulu, HI",USA,21.2720,-157.7750
4,8,Doral Golf Resort and Spa,"Miami, FL",USA,25.8130,-80.3390


In [18]:
courses_meta = {
    "course_key": {
        "description": "DataGolf course ID (primary key)",
        "category": "identifier",
        "quality_issues": "",
    },
    "course_name": {
        "description": "Official course name or layout",
        "category": "identifier",
        "quality_issues": "",
    },
    "location": {
        "description": "City and state/region",
        "category": "geographic",
        "quality_issues": "Sourced from 2024-2026 schedule for recent courses; manually filled for historical venues - format may be inconsistent.",
    },
    "country": {
        "description": "Country",
        "category": "geographic",
        "quality_issues": "Mixed format: schedule-sourced entries use full country name; manual supplement entries use 3-letter ISO codes. Normalize before use.",
    },
    "latitude": {
        "description": "Latitude (decimal degrees)",
        "category": "geographic",
        "quality_issues": "Coordinates for courses absent from the 2024-2026 schedule were filled manually from domain knowledge. Verify before high-stakes use.",
    },
    "longitude": {
        "description": "Longitude (decimal degrees)",
        "category": "geographic",
        "quality_issues": "See latitude note.",
    },
}

section(courses, courses_meta, "courses.csv")


  courses.csv  (6 fields, 180 rows)


Field,Description,Type,Blanks,% Blank,Variable Category,Outliers,% Outlier,Data Quality Issues
course_key,DataGolf course ID (primary key),integer,0,0.00%,identifier,17,9.44%,
course_name,Official course name or layout,string,0,0.00%,identifier,0,0.00%,
location,City and state/region,string,0,0.00%,geographic,0,0.00%,Sourced from 2024-2026 schedule for recent courses; manually filled for historical venues - format may be inconsistent.
country,Country,string,0,0.00%,geographic,0,0.00%,Mixed format: schedule-sourced entries use full country name; manual supplement entries use 3-letter ISO codes. Normalize before use.
latitude,Latitude (decimal degrees),float,0,0.00%,geographic,17,9.44%,Coordinates for courses absent from the 2024-2026 schedule were filled manually from domain knowledge. Verify before high-stakes use.
longitude,Longitude (decimal degrees),float,0,0.00%,geographic,22,12.22%,See latitude note.


## 3. events.csv

In [19]:
events = pd.read_csv(DATA + "events.csv", parse_dates=["start_date", "event_completed"])
events.head()

,event_id_year,event_id,year,season,event_name,tour,start_date,event_completed,course_key,status,winner
0,2_2004,2,2004,2004,Bob Hope Chrysler Classic,pga,NaT,2004-01-25,649,NaN,NaN
1,3_2004,3,2004,2004,FBR Open,pga,NaT,2004-02-01,510,NaN,NaN
2,4_2004,4,2004,2004,Buick Invitational,pga,NaT,2004-02-15,104,NaN,NaN
3,5_2004,5,2004,2004,AT&T Pebble Beach National Pro-Am,pga,NaT,2004-02-08,5,NaN,NaN
4,6_2004,6,2004,2004,Sony Open in Hawaii,pga,NaT,2004-01-18,6,NaN,NaN


In [20]:
events_meta = {
    "event_id_year": {
        "description": "Composite primary key (e.g., '6_2024')",
        "category": "identifier",
        "quality_issues": "",
    },
    "event_id": {
        "description": "DataGolf event ID (reused across years)",
        "category": "identifier",
        "quality_issues": "Not unique alone - always pair with year or use event_id_year.",
    },
    "year": {
        "description": "Calendar year played",
        "category": "temporal",
        "quality_issues": "",
    },
    "season": {
        "description": "PGA Tour season year",
        "category": "temporal",
        "quality_issues": "",
    },
    "event_name": {
        "description": "Official tournament name",
        "category": "tournament metadata",
        "quality_issues": "Same physical event may have different names across years due to sponsorship changes. Use event_id to track recurring events consistently.",
    },
    "tour": {
        "description": "Tour identifier (all 'pga')",
        "category": "tournament metadata",
        "quality_issues": "",
    },
    "start_date": {
        "description": "First round date",
        "category": "temporal",
        "quality_issues": "Null for ~89% of records (pre-2024 events) - sourced only from 2024-2026 schedule files.",
    },
    "event_completed": {
        "description": "Final round date",
        "category": "temporal",
        "quality_issues": "",
    },
    "course_key": {
        "description": "Foreign key to courses.csv (round 1 course)",
        "category": "tournament metadata",
        "quality_issues": "Multi-course events (e.g., Pebble Beach Pro-Am) rotate players across courses by round. Use course_key in rounds.csv for round-level accuracy.",
    },
    "status": {
        "description": "Tournament completion status",
        "category": "tournament metadata",
        "quality_issues": "Null for pre-2024 events.",
    },
    "winner": {
        "description": "Winner name and DataGolf ID",
        "category": "tournament metadata",
        "quality_issues": "Null for pre-2024 events. Not a clean foreign key - requires string parsing to extract dg_id. Derive winner from rounds.csv (fin_text == '1') for full historical coverage.",
    },
}

section(events, events_meta, "events.csv")


  events.csv  (11 fields, 954 rows)


Field,Description,Type,Blanks,% Blank,Variable Category,Outliers,% Outlier,Data Quality Issues
event_id_year,"Composite primary key (e.g., '6_2024')",string,0,0.00%,identifier,954,100.00%,
event_id,DataGolf event ID (reused across years),integer,0,0.00%,identifier,0,0.00%,Not unique alone - always pair with year or use event_id_year.
year,Calendar year played,integer,0,0.00%,temporal,0,0.00%,
season,PGA Tour season year,integer,0,0.00%,temporal,0,0.00%,
event_name,Official tournament name,string,0,0.00%,tournament metadata,229,24.00%,Same physical event may have different names across years due to sponsorship changes. Use event_id to track recurring events consistently.
tour,Tour identifier (all 'pga'),string,0,0.00%,tournament metadata,0,0.00%,
start_date,First round date,string,846,88.68%,temporal,0,0.00%,Null for ~89% of records (pre-2024 events) - sourced only from 2024-2026 schedule files.
event_completed,Final round date,string,0,0.00%,temporal,954,100.00%,
course_key,Foreign key to courses.csv (round 1 course),integer,0,0.00%,tournament metadata,12,1.26%,"Multi-course events (e.g., Pebble Beach Pro-Am) rotate players across courses by round. Use course_key in rounds.csv for round-level accuracy."
status,Tournament completion status,string,846,88.68%,tournament metadata,0,0.00%,Null for pre-2024 events.


## 4. rounds.csv

In [21]:
rounds = pd.read_csv(DATA + "rounds.csv", low_memory=False)
rounds.head()

,event_id_year,event_id,year,dg_id,round_num,course_key,course_par,fin_text,start_hole,teetime,...,scrambling,prox_rgh,prox_fw,great_shots,poor_shots,eagles_or_better,birdies,pars,bogies,doubles_or_worse
0,2_2004,2,2004,20,1,649,72,T9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,6.0,11.0,1.0,0.0
1,2_2004,2,2004,20,2,102,72,T9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,7.0,11.0,0.0,0.0
2,2_2004,2,2004,20,3,202,72,T9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,7.0,7.0,3.0,0.0
3,2_2004,2,2004,20,4,2,72,T9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,5.0,11.0,2.0,0.0
4,2_2004,2,2004,20,5,649,72,T9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,4.0,11.0,1.0,1.0


In [22]:
rounds_meta = {
    "event_id_year": {
        "description": "Foreign key to events.csv",
        "category": "identifier",
        "quality_issues": "",
    },
    "event_id": {
        "description": "DataGolf event ID (not unique alone)",
        "category": "identifier",
        "quality_issues": "Not unique alone - use event_id_year.",
    },
    "year": {
        "description": "Calendar year played",
        "category": "identifier",
        "quality_issues": "",
    },
    "dg_id": {
        "description": "Foreign key to players.csv",
        "category": "identifier",
        "quality_issues": "",
    },
    "round_num": {
        "description": "Round number within the tournament (1-4)",
        "category": "identifier",
        "quality_issues": "",
    },
    "course_key": {
        "description": "Foreign key to courses.csv",
        "category": "identifier",
        "quality_issues": "",
    },
    "course_par": {
        "description": "Course par for the round",
        "category": "scoring",
        "quality_issues": "",
    },
    "fin_text": {
        "description": "Final result (e.g., '1', 'T5', 'CUT', 'WD')",
        "category": "scoring",
        "quality_issues": "Mixed text/numeric-like format - must be parsed before use as a numeric target or feature. CUT players have only 2 round records; rounds 3-4 do not exist for them.",
    },
    "start_hole": {
        "description": "Starting hole (1 = standard, 10 = split tee)",
        "category": "scoring",
        "quality_issues": "55% null - not tracked for older events (pre-~2018). Do not use as a feature without year-filtering.",
    },
    "teetime": {
        "description": "Scheduled tee time (local time)",
        "category": "scoring",
        "quality_issues": "55% null - same coverage gap as start_hole. Stored as string; requires parsing before numeric use.",
    },
    "round_score": {
        "description": "Total strokes for the round",
        "category": "scoring",
        "quality_issues": "Null for WD/DQ players who did not complete the round (~0.2%).",
    },
    "sg_putt": {
        "description": "Strokes gained: putting",
        "category": "strokes gained",
        "quality_issues": "21% null - SG components not tracked before ~2017 for many events. sg_total has full coverage; component breakdown does not.",
    },
    "sg_arg": {
        "description": "Strokes gained: around the green",
        "category": "strokes gained",
        "quality_issues": "21% null - see sg_putt.",
    },
    "sg_app": {
        "description": "Strokes gained: approach",
        "category": "strokes gained",
        "quality_issues": "21% null - see sg_putt.",
    },
    "sg_ott": {
        "description": "Strokes gained: off the tee",
        "category": "strokes gained",
        "quality_issues": "21% null - see sg_putt.",
    },
    "sg_t2g": {
        "description": "Strokes gained: tee to green (excl. putting)",
        "category": "strokes gained",
        "quality_issues": "21% null - see sg_putt. Collinear with components: sg_t2g + sg_putt = sg_total. Do not include all three in a model simultaneously.",
    },
    "sg_total": {
        "description": "Total strokes gained vs. field average",
        "category": "strokes gained",
        "quality_issues": "",
    },
    "driving_dist": {
        "description": "Average driving distance (yards)",
        "category": "shot tracking",
        "quality_issues": "54% null - not tracked for many events without Shotlink data, concentrated in pre-2010 records.",
    },
    "driving_acc": {
        "description": "Driving accuracy (fairways hit rate)",
        "category": "shot tracking",
        "quality_issues": "52% null - see driving_dist.",
    },
    "gir": {
        "description": "Greens in regulation rate",
        "category": "shot tracking",
        "quality_issues": "52% null - see driving_dist.",
    },
    "scrambling": {
        "description": "Scrambling rate (par+ after missed GIR)",
        "category": "shot tracking",
        "quality_issues": "58% null - see driving_dist.",
    },
    "prox_rgh": {
        "description": "Average proximity from rough (feet)",
        "category": "shot tracking",
        "quality_issues": "59% null - see driving_dist. High variance when a player has few rough approach shots in a given round.",
    },
    "prox_fw": {
        "description": "Average proximity from fairway (feet)",
        "category": "shot tracking",
        "quality_issues": "58% null - see driving_dist.",
    },
    "great_shots": {
        "description": "Count of significantly above-average shots",
        "category": "shot tracking",
        "quality_issues": "58% null - see driving_dist. Classification threshold is DataGolf internal and not publicly documented.",
    },
    "poor_shots": {
        "description": "Count of significantly below-average shots",
        "category": "shot tracking",
        "quality_issues": "58% null - see driving_dist. Same threshold caveat as great_shots.",
    },
    "eagles_or_better": {
        "description": "Holes scored eagle or better",
        "category": "hole outcomes",
        "quality_issues": "~0.5% null - minor gap in early records.",
    },
    "birdies": {
        "description": "Holes scored birdie",
        "category": "hole outcomes",
        "quality_issues": "~0.5% null - see eagles_or_better.",
    },
    "pars": {
        "description": "Holes scored par",
        "category": "hole outcomes",
        "quality_issues": "~0.5% null - see eagles_or_better.",
    },
    "bogies": {
        "description": "Holes scored bogey",
        "category": "hole outcomes",
        "quality_issues": "~0.5% null - see eagles_or_better.",
    },
    "doubles_or_worse": {
        "description": "Holes scored double bogey or worse",
        "category": "hole outcomes",
        "quality_issues": "~0.5% null - see eagles_or_better.",
    },
}

section(rounds, rounds_meta, "rounds.csv")


  rounds.csv  (30 fields, 377,363 rows)


Field,Description,Type,Blanks,% Blank,Variable Category,Outliers,% Outlier,Data Quality Issues
event_id_year,Foreign key to events.csv,string,0,0.00%,identifier,377363,100.00%,
event_id,DataGolf event ID (not unique alone),integer,0,0.00%,identifier,0,0.00%,Not unique alone - use event_id_year.
year,Calendar year played,integer,0,0.00%,identifier,0,0.00%,
dg_id,Foreign key to players.csv,integer,0,0.00%,identifier,6102,1.62%,
round_num,Round number within the tournament (1-4),integer,0,0.00%,identifier,0,0.00%,
course_key,Foreign key to courses.csv,integer,0,0.00%,identifier,1059,0.28%,
course_par,Course par for the round,integer,0,0.00%,scoring,0,0.00%,
fin_text,"Final result (e.g., '1', 'T5', 'CUT', 'WD')",string,0,0.00%,scoring,29530,7.83%,Mixed text/numeric-like format - must be parsed before use as a numeric target or feature. CUT players have only 2 round records; rounds 3-4 do not exist for them.
start_hole,"Starting hole (1 = standard, 10 = split tee)",float,208270,55.19%,scoring,0,0.00%,55% null - not tracked for older events (pre-~2018). Do not use as a feature without year-filtering.
teetime,Scheduled tee time (local time),string,208270,55.19%,scoring,95872,25.41%,55% null - same coverage gap as start_hole. Stored as string; requires parsing before numeric use.


## 5. weather.csv

In [23]:
weather = pd.read_csv(DATA + "weather.csv", parse_dates=["date"])
weather.head()

,event_id_year,event_id,year,course_key,date,temperature_mean_c,precipitation_mm,wind_speed_max_kmh,relative_humidity_pct
0,2_2004,2,2004,649,2004-01-20,12.5,1.0,5.6,70.8
1,2_2004,2,2004,649,2004-01-21,13.2,0.3,12.7,50.8
2,2_2004,2,2004,649,2004-01-22,14.3,0.0,15.6,32.7
3,2_2004,2,2004,649,2004-01-23,15.7,0.0,6.2,28.2
4,2_2004,2,2004,649,2004-01-24,12.0,0.4,11.6,58.0


In [24]:
weather_meta = {
    "event_id_year": {
        "description": "Foreign key to events.csv",
        "category": "identifier",
        "quality_issues": "",
    },
    "event_id": {
        "description": "DataGolf event ID",
        "category": "identifier",
        "quality_issues": "",
    },
    "year": {
        "description": "Calendar year",
        "category": "identifier",
        "quality_issues": "",
    },
    "course_key": {
        "description": "Foreign key to courses.csv",
        "category": "identifier",
        "quality_issues": "",
    },
    "date": {
        "description": "Date of observation",
        "category": "temporal",
        "quality_issues": "",
    },
    "temperature_mean_c": {
        "description": "Average air temperature (°C)",
        "category": "atmospheric",
        "quality_issues": "Actual observed weather used for historical records, not the pre-round forecast. Add Gaussian jitter (~±2°C) during training to simulate forecast uncertainty.",
    },
    "precipitation_mm": {
        "description": "Daily precipitation (mm)",
        "category": "atmospheric",
        "quality_issues": "Actual observed - see temperature note. Timing within the day is not captured; a high daily total could be a brief shower or all-day rain.",
    },
    "wind_speed_max_kmh": {
        "description": "Max daily wind speed (km/h)",
        "category": "atmospheric",
        "quality_issues": "Actual observed - see temperature note. Daily maximum does not distinguish sustained wind from gusts; wind direction not included.",
    },
    "relative_humidity_pct": {
        "description": "Average relative humidity (%)",
        "category": "atmospheric",
        "quality_issues": "Actual observed - see temperature note.",
    },
}

section(weather, weather_meta, "weather.csv")


  weather.csv  (9 fields, 5,724 rows)


Field,Description,Type,Blanks,% Blank,Variable Category,Outliers,% Outlier,Data Quality Issues
event_id_year,Foreign key to events.csv,string,0,0.00%,identifier,5724,100.00%,
event_id,DataGolf event ID,integer,0,0.00%,identifier,0,0.00%,
year,Calendar year,integer,0,0.00%,identifier,0,0.00%,
course_key,Foreign key to courses.csv,integer,0,0.00%,identifier,72,1.26%,
date,Date of observation,string,0,0.00%,temporal,5724,100.00%,
temperature_mean_c,Average air temperature (°C),float,0,0.00%,atmospheric,8,0.14%,"Actual observed weather used for historical records, not the pre-round forecast. Add Gaussian jitter (~±2°C) during training to simulate forecast uncertainty."
precipitation_mm,Daily precipitation (mm),float,0,0.00%,atmospheric,948,16.56%,Actual observed - see temperature note. Timing within the day is not captured; a high daily total could be a brief shower or all-day rain.
wind_speed_max_kmh,Max daily wind speed (km/h),float,0,0.00%,atmospheric,157,2.74%,Actual observed - see temperature note. Daily maximum does not distinguish sustained wind from gusts; wind direction not included.
relative_humidity_pct,Average relative humidity (%),float,0,0.00%,atmospheric,267,4.66%,Actual observed - see temperature note.


In [25]:
def raw_summary(df, meta, table_name):
    n = len(df)
    rows = []
    for col in df.columns:
        m = meta.get(col, {})
        null_n = int(df[col].isnull().sum())
        out_n  = outlier_count(df[col])
        dtype  = df[col].dtype
        if pd.api.types.is_float_dtype(dtype):
            t = "float"
        elif pd.api.types.is_integer_dtype(dtype):
            t = "integer"
        elif pd.api.types.is_bool_dtype(dtype):
            t = "boolean"
        else:
            t = "string"
        rows.append({
            "Table":               table_name,
            "Field":               col,
            "Description":         m.get("description", ""),
            "Type":                t,
            "Blanks":              null_n,
            "% Blank":             round(null_n / n, 4),
            "Variable Category":   m.get("category", ""),
            "Outliers":            out_n,
            "% Outlier":           round(out_n / n, 4),
            "Data Quality Issues": m.get("quality_issues", ""),
        })
    return pd.DataFrame(rows)

tables = [
    (players, players_meta, "players.csv"),
    (courses, courses_meta, "courses.csv"),
    (events,  events_meta,  "events.csv"),
    (rounds,  rounds_meta,  "rounds.csv"),
    (weather, weather_meta, "weather.csv"),
]

combined = pd.concat([raw_summary(df, meta, name) for df, meta, name in tables], ignore_index=True)
combined.to_csv("../reports/data_dictionary.csv", index=False)
print(f"Exported {len(combined)} rows to reports/data_dictionary.csv")

Exported 61 rows to reports/data_dictionary.csv
